# WeldVision X5 — YOLOv8 Training Notebook

**Pipeline:** Roboflow (Label) → Train (Colab or Local) → R2 (Upload) → GitHub Actions (BPU Compile) → RDK X5 (Deploy)

This notebook automatically detects if you are running in Google Colab (Cloud) or locally (VS Code/Jupyter) and adjusts save paths accordingly.

In [ ]:
%pip uninstall torch torchvision -y
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121




In [ ]:
# ── Cell 1: Setup Environment (Colab vs Local) ─────────
import os, sys
IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/WeldVision'
else:
    print("Running locally (VS Code / Jupyter)")
    SAVE_DIR = './WeldVision_Output'  # type: ignore[misc]

os.makedirs(SAVE_DIR, exist_ok=True)
print(f'✓ Models will be saved to: {SAVE_DIR}')

In [ ]:
# ── Cell 2: Install dependencies + Download dataset from Roboflow ─────────────
if IS_COLAB:
    %pip install -q ultralytics roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="77VZXPv0TIQSTIjXQDVP")
project = rf.workspace("jwai").project("weldvision-ribcd")
version = project.version(6)
dataset = version.download("yolov8")

In [ ]:
# ── Cell 3: Train YOLOv8 ──────────────────────────────────────────────────────
import torch
from ultralytics import YOLO

model = YOLO('yolov8s.pt')  # yolov8s.pt is optimal for T1000 8GB / RDK X5 Edge

# Check if NVIDIA GPU is actually available (PyTorch + Drivers configured correctly)
has_gpu = torch.cuda.is_available()
if not has_gpu:
    print("\n⚠️ WARNING: PyTorch cannot detect your NVIDIA GPU!")
    if IS_COLAB:
        print("-> Fix: Go to top menu 'Runtime' > 'Change runtime type' > Select 'T4 GPU'.")
    else:
        print("-> Fix: You installed the CPU version of PyTorch. Reinstall with CUDA:")
        print("   pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121")
    print("Training will proceed on CPU, but it will be EXTREMELY slow.\n")

# Optimized parameters for 8GB VRAM GPUs (like NVIDIA T1000 8GB) 
results = model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,       # 16 is perfect for 8GB VRAM (drop to 8 if OOM)
    device=0 if has_gpu else 'cpu', # Auto-fallback to CPU to prevent crash
    amp=True,       # Enable Automatic Mixed Precision for speed
    workers=4,      # Optimized CPU worker count for stability
    project=SAVE_DIR,  # Save dynamically to Colab Drive or Local folder
    name='train'
)
print('✓ Training complete')

In [ ]:
# ── Cell 4: Export best weights → ONNX + extract metadata ───────────────────────
import shutil, os, json, pandas as pd
from ultralytics import YOLO

# Load best weights
best_pt = f'{SAVE_DIR}/train/weights/best.pt'
if not os.path.exists(best_pt):
    best_pt = 'runs/detect/train/weights/best.pt'  # Fallback

print(f'Loading: {best_pt}')
model = YOLO(best_pt)

# Verify ONNX input node name = 'images' (required by Horizon BPU compiler)
model.export(format='onnx', dynamic=False, simplify=True)

import onnx  # type: ignore
m = onnx.load('best.onnx')
input_name = m.graph.input[0].name
print(f'ONNX input node name: "{input_name}"')
if input_name != 'images':
    print('⚠️  WARNING: Input node is not "images" — hb_mapper may reject this model!')
else:
    print('✓ Input node name is correct')

# Extract Metadata from results.csv
csv_path = f'{SAVE_DIR}/train/results.csv'
if not os.path.exists(csv_path):
    csv_path = 'runs/detect/train/results.csv'  # Fallback

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    best_row = df.loc[df['metrics/mAP50-95(B)'].idxmax()]
    metadata = {
        'map50': round(float(best_row['metrics/mAP50(B)']), 4),  # type: ignore[arg-type]
        'map50_95': round(float(best_row['metrics/mAP50-95(B)']), 4),  # type: ignore[arg-type]
        'epochs': len(df),
        'dataset_version': 'Roboflow v5'
    }
    with open('metadata.json', 'w') as f:
        json.dump(metadata, f, indent=2)
    print('\n✓ Extracted Metadata:')
    print(json.dumps(metadata, indent=2))

# Save to final directory
shutil.copy('best.onnx', f'{SAVE_DIR}/best.onnx')
if os.path.exists('metadata.json'):
    shutil.copy('metadata.json', f'{SAVE_DIR}/metadata.json')
print(f'\n✓ Saved best.onnx and metadata.json to {SAVE_DIR}')
print('Next: Upload to WeldVision MLOps → Model Registry')

In [ ]:
# ── Cell 5: Download files to your computer ───────────────────────────────
import os

if IS_COLAB:
    from google.colab import files  # type: ignore
    files.download('best.onnx')
    if os.path.exists('metadata.json'):
        files.download('metadata.json')
else:
    print(f"Files are ready locally in: {os.path.abspath(SAVE_DIR)}")